# The Heritage Short — Music Score Generator (ACE-Step 1.5) — Google Colab

**Colab settings:** Runtime -> Change runtime type -> **GPU (T4)**. No API key is needed; model weights are public.

Colab twin of `notebooks/acestep_music.ipynb`. It produces the **same contract** for `py run.py music --import-dir <folder>`: `candidate_01_seed777.wav`, `candidate_02_seed778.wav`, `candidate_03_seed779.wav` plus `music_run_metadata.json`. Extract the downloaded zip into `outputs/music/`.

Uses the tested pinned trio (`transformers==4.57.6`, `huggingface_hub==0.36.2`, `diffusers==0.35.2`) and does **not** run `pip install -e` (it failed twice). A real load assertion guards against a false 'Models initialized.' message. **Second fresh-session fix:** the DiT model file imports `vector_quantize_pytorch.ResidualFSQ`; the install cell now adds `vector-quantize-pytorch>=1.27.15`, and cells 5-6 fail with a clear message if cell 4 never verified the model (FAILURES.md 024).

In [ ]:
# 1. Clone ACE-Step 1.5 if needed, then install the tested compatibility trio + VQ module.
# transformers 5.x is outside ACE-Step's >=4.51,<4.58 range; downgrading it
# broke diffusers 0.42.0 (resolve_revision needs hub 1.26.0), but
# transformers 4.57.x needs hub <1.0. These three pins work together.
!test -d /content/ACE-Step-1.5 || git clone --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git /content/ACE-Step-1.5
!pip install -q "transformers==4.57.6" "huggingface_hub==0.36.2" "diffusers==0.35.2" soundfile accelerate einops loguru safetensors "vector-quantize-pytorch>=1.27.15"

In [ ]:
# 2. Fail loudly here if the dependency chain regressed.
from importlib.metadata import version
from diffusers.models import AutoencoderOobleck  # the exact import that broke before
from vector_quantize_pytorch import ResidualFSQ  # missing package that broke the DiT load

for pkg, expected in (
    ("transformers", "4.57.6"),
    ("huggingface_hub", "0.36.2"),
    ("diffusers", "0.35.2"),
):
    got = version(pkg)
    print(f"{pkg}: {got}")
    assert got == expected, f"{pkg} {got} != {expected}; see FAILURES.md entry 017"
print("Compatibility check passed: AutoencoderOobleck and ResidualFSQ import.")

In [ ]:
# 3. Music spec (copied directly from storyboard.json -> music)
import json, time, shutil
from pathlib import Path

MUSIC_SPEC = {
    "model": "ACE-Step/Ace-Step1.5",
    "dit_config": "acestep-v15-turbo",
    "lm_model": "acestep-5Hz-lm-0.6B",
    "caption": "Warm acoustic Ethiopian-inspired folk instrumental with gentle krar strings, soft bamboo flute, and subtle felt piano, unhurried morning courtyard mood, organic warm texture, no vocals",
    "style": "acoustic folk instrumental",
    "mood": "warm, grateful, unhurried, nostalgic",
    "instruments": "krar plucked strings, soft bamboo flute, soft felt piano",
    "texture": "warm organic acoustic",
    "bpm": 72,
    "keyscale": "D major",
    "duration": 60.0,
    "base_seed": 777,
    "num_candidates": 3,
}

OUT_DIR = Path("/content/music_candidates")
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Safe defaults set BEFORE any model call. If cell 4 aborts, later cells
# give a clear error instead of a confusing NameError (FAILURES.md 024).
MODEL_READY = False
llm_handler = None
llm_initialized = False
gpu = gpu_total_gb = gpu_used_gb = model_load_sec = None
records = []
run_info = {}
print("Ready. Output directory:", OUT_DIR)

In [ ]:
# 4. Initialize ACE-Step 1.5 and ASSERT that the DiT model really loaded.
import sys, torch
PROJECT_ROOT = "/content/ACE-Step-1.5"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"
sys.path.insert(0, PROJECT_ROOT)  # replaces `pip install -e .`, which failed twice

from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

assert torch.cuda.is_available(), "GPU required: set Colab Runtime = GPU T4"
gpu = torch.cuda.get_device_properties(0)
gpu_total_gb = round(gpu.total_memory / 1024**3, 1)
print(f"GPU detected: {gpu.name} ({gpu_total_gb} GB)")

t_load = time.perf_counter()
dit_handler = AceStepHandler()
dit_handler.initialize_service(
    project_root=PROJECT_ROOT,
    config_path=MUSIC_SPEC["dit_config"],
    device="cuda",
)
model_load_sec = time.perf_counter() - t_load
assert getattr(dit_handler, "model", None) is not None, (
    "DiT model failed to load (dit_handler.model is None). "
    "Do not print 'Models initialized.' when loading failed."
)
gpu_used_gb = round(torch.cuda.memory_allocated() / 1024**3, 1)
print(f"DiT loaded in {model_load_sec:.1f} s; GPU memory after load ~{gpu_used_gb} GB")
MODEL_READY = True

# Language-model planner is OPTIONAL: generate_music skips it when
# llm_initialized is False, so instrumental generation still works.
llm_handler = None
llm_initialized = False
try:
    llm_candidate = LLMHandler()
    llm_candidate.initialize(
        checkpoint_dir=CHECKPOINT_DIR,
        lm_model_path=MUSIC_SPEC["lm_model"],
        backend="pt",
        device="cuda",
    )
    llm_initialized = bool(getattr(llm_candidate, "llm_initialized", False))
    if llm_initialized:
        llm_handler = llm_candidate
except Exception as exc:
    print("Optional LM planner failed; continuing without it:", type(exc).__name__, str(exc)[:300])
print("llm_initialized =", llm_initialized)
print("Models initialized (DiT verified). ")

In [ ]:
# 5. Generate 3 instrumental candidates sequentially (fixed seeds 777, 778, 779)
if not MODEL_READY:
    raise RuntimeError('DiT model was not verified in cell 4. Fix the install, rerun cell 4, then continue; no candidates were generated.')
records = []
for idx in range(MUSIC_SPEC["num_candidates"]):
    seed = MUSIC_SPEC["base_seed"] + idx
    t0 = time.perf_counter()
    params = GenerationParams(
        task_type="text2music",
        caption=MUSIC_SPEC["caption"],
        lyrics="",
        instrumental=True,
        bpm=MUSIC_SPEC["bpm"],
        keyscale=MUSIC_SPEC["keyscale"],
        duration=MUSIC_SPEC["duration"],
        inference_steps=8,
        seed=seed,
    )
    config = GenerationConfig(
        batch_size=1,
        use_random_seed=False,
        seeds=[seed],
        audio_format="wav",
    )
    result = generate_music(
        dit_handler,
        llm_handler,
        params,
        config,
        save_dir=str(OUT_DIR / f"tmp_{idx+1}"),
    )
    latency_ms = int((time.perf_counter() - t0) * 1000)
    if not result.success or not result.audios:
        raise RuntimeError(f"Candidate {idx+1} (seed={seed}) failed: {result.error}")

    src_path = Path(result.audios[0]["path"])
    dst_name = f"candidate_{idx+1:02d}_seed{seed}.wav"
    dst_path = OUT_DIR / dst_name
    shutil.copy2(src_path, dst_path)
    records.append({
        "candidate_id": f"candidate_{idx+1:02d}",
        "file_name": dst_name,
        "model": MUSIC_SPEC["model"],
        "dit_config": MUSIC_SPEC["dit_config"],
        "seed": seed,
        "bpm": MUSIC_SPEC["bpm"],
        "keyscale": MUSIC_SPEC["keyscale"],
        "duration_sec": MUSIC_SPEC["duration"],
        "latency_ms": latency_ms,
        "status": "ok",
    })
    print(f"Saved {dst_name} in {latency_ms} ms")

run_info = {
    "gpu": gpu.name,
    "gpu_total_gb": gpu_total_gb,
    "gpu_used_after_load_gb": gpu_used_gb,
    "model_load_sec": round(model_load_sec, 2),
    "llm_initialized": llm_initialized,
}
meta_path = OUT_DIR / "music_run_metadata.json"
meta_path.write_text(
    json.dumps({"spec": MUSIC_SPEC, "candidates": records, "run_info": run_info}, indent=2),
    encoding="utf-8",
)
print("Wrote", meta_path)

In [ ]:
# 6. Package and download the same zip contract as the Kaggle notebook.
from google.colab import files
if len(records) != MUSIC_SPEC['num_candidates']:
    raise RuntimeError('Cell 5 did not finish all 3 candidates; nothing was packaged.')

zip_path = shutil.make_archive("/content/heritage_music_candidates", "zip", OUT_DIR)
print("Downloaded zip: extract into outputs/music/ in the repo.", zip_path)
for rec in records:
    print(rec["file_name"], f"({rec['latency_ms']} ms)")
print("run_info:", run_info)
files.download(zip_path)